In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [53]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [67]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex".
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "````json")
    text = chat(messages, stop_sequences=["````"])
    return json.loads(text)

In [68]:
dataset = generate_dataset()
dataset
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

dataset

[{'task': 'Extract AWS account ID from an ARN string',
  'format': 'regex',
  'solution_criteria': "Regex pattern should correctly extract the 12-digit account ID from valid ARNs (e.g., 'arn:aws:s3:::bucket-name' or 'arn:aws:iam::123456789012:role/MyRole')"},
 {'task': 'Parse an AWS CloudWatch alarm configuration and return the alarm name and threshold value',
  'format': 'python',
  'solution_criteria': "Function should accept a JSON string or dict containing CloudWatch alarm config and return a dict with 'alarm_name' and 'threshold' keys. Should handle missing fields gracefully."},
 {'task': 'Create a JSON IAM policy document that allows read-only access to a specific S3 bucket',
  'format': 'json',
  'solution_criteria': "Valid JSON policy document with proper Statement array, Effect set to 'Allow', Actions limited to s3:GetObject and s3:ListBucket, and Resource restricted to a specific bucket ARN and objects within it"}]

In [60]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then return the result"""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criterial you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Output Format:  
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [62]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [63]:
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [64]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade.get("score", 0)
    reasoning = model_grade.get("reasoning", "")
    
    syntax_score= grade_syntax(output, test_case)
    score = (model_score+syntax_score)/2
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }
    

In [65]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results
  

In [66]:
dataset
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval (dataset)

Average score: 5.0


In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\nfrom typing import Tuple\n\ndef validate_s3_bucket_name(bucket_name: str) -> Tuple[bool, str]:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens (-)\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        Tuple[bool, str]: (is_valid, error_message)\n            - is_valid: True if bucket name is valid, False otherwise\n            - error_message: Description of validation error (empty string if valid)\n    \"\"\"\n    \n    # Check if input is a string\n    if not isinstance(buc